In [1]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import DataStructs

In [5]:
def load_smiles(file_path):
    df = pd.read_excel(file_path) 
    # 假设列名是 SMILES，如果不是改成你自己的列名
    return df['SMILES'].dropna().tolist()

train_smiles = load_smiles("STING抑制剂SMILES.xlsx")
tcm_smiles = load_smiles("中药预测SMILES.xlsx")

In [6]:
def smiles_to_mols(smiles_list):
    mols = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol:
            mols.append(mol)
    return mols

train_mols = smiles_to_mols(train_smiles)
tcm_mols = smiles_to_mols(tcm_smiles)

[00:15:28] Explicit valence for atom # 45 N, 4, is greater than permitted
[00:15:28] Explicit valence for atom # 41 N, 4, is greater than permitted
[00:15:28] Explicit valence for atom # 21 N, 4, is greater than permitted
[00:15:28] Explicit valence for atom # 23 N, 4, is greater than permitted
[00:15:28] Explicit valence for atom # 16 N, 4, is greater than permitted
[00:15:28] Explicit valence for atom # 16 N, 4, is greater than permitted
[00:15:28] Explicit valence for atom # 16 N, 4, is greater than permitted
[00:15:28] Explicit valence for atom # 16 N, 4, is greater than permitted
[00:15:28] Can't kekulize mol.  Unkekulized atoms: 35 36 37 38 39
[00:15:29] Explicit valence for atom # 37 C, 5, is greater than permitted
[00:15:29] Explicit valence for atom # 8 N, 4, is greater than permitted
[00:15:29] Explicit valence for atom # 5 N, 4, is greater than permitted
[00:15:29] Explicit valence for atom # 5 N, 4, is greater than permitted
[00:15:29] Explicit valence for atom # 8 N, 4, is

In [7]:
def get_ecfp4_fps(mols):
    fps = []
    for mol in mols:
        # radius=2 = ECFP4, nBits=1024/2048 论文常用
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024)
        fps.append(fp)
    return fps

train_fps = get_ecfp4_fps(train_mols)
tcm_fps = get_ecfp4_fps(tcm_mols)

In [12]:
def max_tanimoto(query_fp, ref_fps):
    sims = [DataStructs.TanimotoSimilarity(query_fp, fp) for fp in ref_fps]
    return max(sims)

# 计算每个 TCM 分子与训练集的最大相似度
tcm_max_similarities = [max_tanimoto(fp, train_fps) for fp in tcm_fps]

mean_sim = np.mean(tcm_max_similarities)
median_sim = np.median(tcm_max_similarities)
in_ad = sum(1 for s in tcm_max_similarities if s >= 0.4)  # 常用AD阈值
total = len(tcm_max_similarities)

print(f"平均最大相似度: {mean_sim:.2f}")
print(f"中位数相似度: {median_sim:.2f}")
print(f"适用域内化合物数量: {in_ad}/{total} ({in_ad/total*100:.1f}%)")

# 保存结果到 CSV（可画图）
result_df = pd.DataFrame({
    "TCM_SMILES": [Chem.MolToSmiles(mol) for mol in tcm_mols],
    "Max_Similarity_To_Training": tcm_max_similarities,
    "In_Applicability_Domain": [s >= 0.45 for s in tcm_max_similarities]
})
result_df.to_excel("tcm_applicability_domain_result.xlsx", index=False)

平均最大相似度: 0.21
中位数相似度: 0.20
适用域内化合物数量: 7/1588 (0.4%)
